# H100 네 장에서 Gemma 4와 DFlash 비교하기

Gemma 4 31B IT 단독 실행과 같은 모델에 DFlash를 추가한 실행을 비교합니다. 같은 vLLM 서버 설정에 대해 질문하고, 동시 요청 수가 1건일 때와 4건일 때 각각 응답 시간과 처리량을 잽니다.

**발표 준비**　H100 네 장이 있는 Linux 서버에서 1~6절을 순서대로 실행합니다. 두 설정의 결과를 얻으면 7절도 한 번 실행하고, 출력이 보이도록 노트북을 저장합니다. 현장에서 요청을 보내려면 DFlash 서버와 노트북 커널을 켜 둡니다.

**발표 당일**　[7절 · 10분 시연](#demo)만 엽니다. 서버 설치, 모델 로딩, 전체 부하 측정은 발표 전에 끝냅니다. [발표 자료](https://teimakes.github.io/ai-cookbook/#lab)

## 1. 실행 환경 준비

이미 두 모델을 지원하는 vLLM 환경이 있다면 그 환경에서 JupyterLab을 엽니다. 새 환경을 만들 때는 아래처럼 Z Lab이 모델 카드에서 안내한 vLLM 소스를 사용합니다. 이 설치는 소스 빌드이므로 CUDA를 지원하는 Linux 빌드 환경과 CUDA Toolkit이 필요합니다. 기존 Qwen 실습 환경과는 별도로 만드세요.

```bash
uv venv --python 3.12
source .venv/bin/activate
uv pip install --torch-backend=auto "vllm @ git+https://github.com/vllm-project/vllm.git@refs/pull/41703/head" jupyterlab pandas
jupyter lab
```

노트북에서 두 번 멈춥니다. **3절**에서는 터미널에서 Gemma 서버를 켭니다. **5절**에서는 그 서버를 종료한 뒤 DFlash와 함께 다시 실행합니다. [Z Lab 설치·실행 예시](https://huggingface.co/z-lab/gemma-4-31B-it-DFlash#quick-start) · [vLLM 소스 빌드](https://docs.vllm.ai/en/stable/getting_started/installation/gpu/)

## 2. 모델 폴더와 GPU 확인

`TARGET`에 Gemma 4 31B IT의 폴더를, `DRAFT`에 이 모델용 Z Lab DFlash의 폴더를 지정합니다. 설정 파일에서 층 수를, 색인에서 가중치 파일 이름을 읽습니다. GPU 목록에는 할당받은 H100 네 장이 보여야 합니다. [발표의 모델 파일 설명](https://teimakes.github.io/ai-cookbook/#files)

In [ ]:
import json
import shlex
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from time import perf_counter

import pandas as pd
import requests
from openai import OpenAI
from prometheus_client.parser import text_string_to_metric_families

TARGET = Path("/models/gemma-4-31B-it")
DRAFT = Path("/models/gemma-4-31B-it-DFlash")

print(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
    text=True,
))
target_config = json.loads((TARGET / "config.json").read_text())
draft_config = json.loads((DRAFT / "config.json").read_text())
index = json.loads((TARGET / "model.safetensors.index.json").read_text())
print("Gemma 텍스트 모델 층 수", target_config["text_config"]["num_hidden_layers"])
print("가중치 파일", sorted(set(index["weight_map"].values())))
print("보조 모델", draft_config["architectures"], "블록 크기", draft_config["block_size"])

## 3. Gemma 서버 실행

다음 셀은 서버 실행 명령을 **출력**합니다. 그 한 줄을 같은 서버의 별도 터미널에서 실행합니다. 터미널에 `Application startup complete`가 표시되면 서버를 켜 둔 채 노트북으로 돌아옵니다.

- `--tensor-parallel-size 4`는 네 GPU가 모델 하나의 계산을 나누도록 합니다.
- `--max-model-len 4096`은 한 요청에서 입력과 출력을 합해 사용할 수 있는 최대 토큰 수입니다.
- 두 설정 모두 텍스트만 처리하고, 반복되는 입력의 계산을 매번 다시 합니다. 아래 명령은 1절의 Z Lab 환경용입니다. 이미 설치한 다른 버전에서 접두사 캐싱 옵션이 다르면 `vllm serve --help`에 나오는 `--disable-prefix-caching`으로 바꿉니다. 이 실습은 동시 요청 수가 바뀔 때도 같은 방식으로 계산하도록 CUDA 그래프를 끕니다(`--enforce-eager`). 이 설정에서 얻은 속도를 비교합니다. [모델 작성자의 배치 검증 논의](https://github.com/vllm-project/vllm/pull/41703) · [발표의 GPU 구성](https://teimakes.github.io/ai-cookbook/#parallel)

In [ ]:
base_url = "http://127.0.0.1:8000"
client = OpenAI(base_url=f"{base_url}/v1", api_key="local")

server = [
    sys.executable, "-m", "vllm.entrypoints.cli.main", "serve", str(TARGET),
    "--host", "127.0.0.1", "--tensor-parallel-size", "4",
    "--dtype", "bfloat16", "--max-model-len", "4096",
    "--language-model-only", "--no-enable-prefix-caching",
    "--attention-backend", "TRITON_ATTN", "--enforce-eager",
]
print(shlex.join(server))

### 질문 한 건의 응답 시간 측정

아래 셀은 실제 vLLM 서버 실행 명령을 모델에 보여 주고 두 옵션의 뜻을 묻습니다. 답에서 `--tensor-parallel-size 4`가 **모델 계산을 네 GPU에 나눈다**는 뜻인지, `--max-model-len 4096`이 **한 요청의 입력과 출력 토큰 수를 합한 상한**인지 확인합니다.

노트북은 응답을 조금씩 받아 첫 글이 도착한 시각과 응답이 끝난 시각을 잽니다. 한 번에 여러 토큰이 도착할 수 있으므로 첫 응답 시간은 클라이언트에서 처음 글을 받은 시점을 기준으로 합니다.

In [ ]:
def ask(question):
    start = perf_counter()
    stream = client.chat.completions.create(
        model=str(TARGET),
        messages=[{"role": "user", "content": question}],
        temperature=0,
        max_tokens=128,
        stream=True,
        stream_options={"include_usage": True},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    text_parts = []
    first_ms = None
    finish = None
    for chunk in stream:
        if chunk.usage:
            usage = chunk.usage
        if chunk.choices:
            choice = chunk.choices[0]
            if choice.delta.content:
                if not text_parts:
                    first_ms = (perf_counter() - start) * 1000
                text_parts.append(choice.delta.content)
            if choice.finish_reason:
                finish = choice.finish_reason

    elapsed_ms = (perf_counter() - start) * 1000
    tokens = usage.completion_tokens
    return {
        "답변": "".join(text_parts), "종료": finish,
        "입력 토큰": usage.prompt_tokens, "출력 토큰": tokens,
        "첫 응답 ms": first_ms, "전체 ms": elapsed_ms,
        "TPOT ms": (elapsed_ms - first_ms) / (tokens - 1)
        if tokens > 1 and first_ms is not None else None,
    }


check_question = (
    f"{shlex.join(server)}\n"
    "이 실행 명령에서 --tensor-parallel-size 4와 --max-model-len 4096은 각각 무엇을 뜻하나요? 두 문장으로 설명해 주세요."
)
baseline_answer = ask(check_question)
print(baseline_answer["답변"])

## 4. 같은 질문으로 Gemma 단독 성능 재기

서버 설정과 성능 측정에 관한 여덟 질문을 세 번씩 보냅니다. **동시 요청 1건**에서는 앞의 답이 끝난 뒤 다음 요청을 보냅니다. **4건**에서는 동시에 최대 네 건의 답을 기다립니다. 각 조건에서 먼저 보내는 연습 요청은 측정에서 뺍니다.

| 출력 열 | 읽는 방법 |
| --- | --- |
| 출력 토큰/초 | 24건의 총 출력 토큰 수를 전체 측정 시간(초)으로 나눕니다. |
| 첫 응답 p50 ms | 요청 전송부터 첫 글을 받을 때까지의 시간 중 가운데 값입니다. |
| TPOT p50 ms | 각 요청에서 (전체 시간 − 첫 응답 시간) ÷ (출력 토큰 수 − 1)을 구한 뒤 가운데 값을 봅니다. 출력이 한 토큰이면 계산에서 제외합니다. |
| 전체 p95 ms | 전체 응답 시간의 95번째 백분위수입니다. 느린 요청이 얼마나 걸렸는지 봅니다. |

이 요청들은 문장 끝에서 생성을 마치므로 실제 출력 길이가 서로 다를 수 있습니다. 두 설정의 토큰 수도 함께 봅니다. 24건은 실습용으로 적은 수여서 p95는 몇 건의 느린 요청에도 달라질 수 있습니다. [발표의 시간 계산 예](https://teimakes.github.io/ai-cookbook/#metrics)

In [ ]:
questions = [
    check_question,
    "vLLM의 --tensor-parallel-size와 --max-num-seqs는 각각 무엇을 정하나요?",
    "답변이 출력되기까지 오래 걸리는데 일단 출력이 시작되면 빠릅니다. 서버에서 어떤 시간을 먼저 확인해야 하나요?",
    "동시 요청 상한을 늘리자 초당 출력 토큰 수와 p95 응답 시간이 모두 늘었습니다. 두 수치가 함께 늘 수 있는 이유를 설명해 주세요.",
    "두 서버의 처리량을 비교하려고 합니다. 질문과 답변 길이를 어떻게 맞춰야 하나요?",
    "언어 모델의 부하 테스트에서 첫 실행을 따로 측정하는 이유가 무엇인가요?",
    "서버가 반환한 출력 토큰 수로 초당 처리량을 구하는 파이썬 코드를 보여 주세요.",
    "두 요청의 입력 앞부분이 같으면 추론 엔진이 어떤 계산을 재사용할 수 있나요?",
]

def measure(concurrency, counters=None):
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        list(pool.map(ask, questions[:concurrency]))
        if counters:
            before = counters()
        start = perf_counter()
        answers = list(pool.map(ask, questions * 3))
        seconds = perf_counter() - start
        if counters:
            after = counters()

    rows = pd.DataFrame(answers)
    summary = {
        "동시 요청": concurrency, "요청 수": len(rows),
        "입력 토큰 합": rows["입력 토큰"].sum(), "출력 토큰 합": rows["출력 토큰"].sum(),
        "출력 토큰/초": rows["출력 토큰"].sum() / seconds,
        "첫 응답 p50 ms": rows["첫 응답 ms"].median(),
        "TPOT p50 ms": rows["TPOT ms"].median(), "전체 p95 ms": rows["전체 ms"].quantile(.95),
    }
    if counters:
        proposed = after["제안"] - before["제안"]
        accepted = after["채택"] - before["채택"]
        summary["제안 토큰"] = proposed
        summary["채택 토큰"] = accepted
        summary["채택률 %"] = 100 * accepted / proposed if proposed else None
    return summary

In [ ]:
baseline = pd.DataFrame([measure(1), measure(4)]).set_index("동시 요청")
baseline.round(2)

## 5. DFlash를 켜고 같은 질문 다시 보내기

Gemma 서버를 실행한 **터미널에서 Ctrl+C**를 누르고 종료될 때까지 기다립니다. **노트북은 계속 열어 둡니다.** 아래 셀이 출력한 명령을 같은 터미널에서 실행합니다.

Z Lab 모델의 블록 크기는 16입니다. 이미 정해진 토큰 한 자리와 새로 제안할 자리 15개를 함께 계산합니다. 예를 들어 후보 세 개 중 앞의 두 개만 맞으면 그 두 개를 채택하고 세 번째 위치는 Gemma가 고른 토큰을 씁니다. 이렇게 하면 맞은 토큰 수만큼 Gemma 단독 실행에서 필요했던 계산 횟수를 줄일 수 있습니다. 후보를 만드는 데도 시간과 메모리가 들기 때문에 실제 효과는 6절에서 측정합니다. [발표의 토큰별 검사 예](https://teimakes.github.io/ai-cookbook/#verify)

In [ ]:
speculation = {
    "method": "dflash",
    "model": str(DRAFT),
    "num_speculative_tokens": draft_config["block_size"] - 1,
    "attention_backend": "FLASH_ATTN",
}
print(shlex.join(server + ["--speculative-config", json.dumps(speculation)]))

서버가 준비되면 아래 셀을 실행합니다. 두 답변에서 GPU 수와 모델 길이에 대한 설명을 비교합니다. 종료 사유가 `length`이면 지정한 출력 길이에 도달한 것이므로 답이 중간에 끊겼는지도 확인합니다.

In [ ]:
dflash_answer = ask(check_question)
pd.DataFrame([baseline_answer, dflash_answer], index=["Gemma 단독", "DFlash"])[
    ["답변", "종료", "출력 토큰"]
]

## 6. DFlash 후보 중 실제로 쓴 토큰 세기

서버의 `/metrics` 주소에서 지금까지 제안한 토큰 수와 채택한 토큰 수를 읽습니다. 실습 시작 전후의 차이를 구하면 이번 요청에 해당하는 수를 얻습니다. 이때는 같은 서버에 다른 요청을 보내지 않습니다. 예를 들어 100개를 제안하고 70개를 썼다면 채택률은 70%입니다.

In [ ]:
def draft_counts():
    response = requests.get(f"{base_url}/metrics")
    response.raise_for_status()
    names = {
        "vllm:spec_decode_num_draft_tokens_total": "제안",
        "vllm:spec_decode_num_accepted_tokens_total": "채택",
    }
    counts = {"제안": 0, "채택": 0}
    for family in text_string_to_metric_families(response.text):
        for sample in family.samples:
            if sample.name in names:
                counts[names[sample.name]] += int(sample.value)
    return counts

In [ ]:
dflash = pd.DataFrame([measure(1, draft_counts), measure(4, draft_counts)]).set_index("동시 요청")
dflash.round(2)

**제안 토큰**이 0이면 이 측정으로는 DFlash의 동작을 확인할 수 없습니다. 서버를 시작한 터미널에서 보조 모델이 로드됐는지, `/metrics`에 두 카운터가 있는지 확인합니다. 다음 표에서 **처리량 배수**가 1보다 크면 초당 출력이 늘어난 것입니다. **TPOT 감소율**이 양수이면 첫 응답 후 토큰당 평균 시간이 줄어든 것입니다.

In [ ]:
comparison = pd.DataFrame({
    "단독 토큰/초": baseline["출력 토큰/초"],
    "DFlash 토큰/초": dflash["출력 토큰/초"],
    "처리량 배수": dflash["출력 토큰/초"] / baseline["출력 토큰/초"],
    "TPOT 감소율 %": 100 * (1 - dflash["TPOT p50 ms"] / baseline["TPOT p50 ms"]),
    "단독 전체 p95 ms": baseline["전체 p95 ms"],
    "DFlash 전체 p95 ms": dflash["전체 p95 ms"],
    "채택률 %": dflash["채택률 %"],
})
comparison.round(2)

### 측정값 확인

동시 요청 수가 같은 행끼리 비교합니다. 출력 길이가 다르면 응답 시간 차이를 해석할 때 두 설정의 출력 토큰 합도 봅니다. 단독 실행과 DFlash 양쪽의 표를 얻었다면 아래 7절을 실행해 발표에 쓸 출력을 저장합니다.

<a id="demo"></a>

## 7. 발표용 10분 시연

발표 전에 이 절까지 실행하고 출력을 저장합니다. 현장에서 서버 응답이 늦어지면 5절에 저장한 답변 비교표와 아래의 측정 결과로 설명을 이어갑니다.

### 0~2분 · 두 실행의 차이

두 설정에서 같은 Gemma를 같은 네 GPU에 올렸습니다. 아래에서 대상 모델, 추가한 보조 모델, 한 번에 제안할 수 있는 후보 수를 확인합니다.

In [ ]:
print("대상 모델", TARGET.name)
print("DFlash 실행에서 추가한 모델", DRAFT.name)
print("한 번에 제안할 수 있는 토큰 수", speculation["num_speculative_tokens"])

### 2~5분 · 같은 질문에 대한 답변

DFlash 서버에 질문을 한 번 보냅니다. 첫 행은 3절에서 받아 둔 Gemma 단독 답변입니다. 두 번째 행은 방금 받은 답변입니다. GPU 수와 토큰 수 상한을 각각 정확하게 설명했는지 확인합니다. 이 한 건은 부하 테스트 결과에 포함되지 않습니다.

In [ ]:
print(check_question)
live_answer = ask(check_question)
answers = pd.DataFrame([baseline_answer, live_answer], index=["Gemma 단독", "DFlash"])
with pd.option_context("display.max_colwidth", None):
    display(answers[["답변", "종료", "출력 토큰"]])

### 5~9분 · 채택률, 처리량, 응답 시간

아래 두 셀은 4절과 6절에서 측정한 값을 표시합니다. 먼저 DFlash가 실제로 몇 개를 제안했고 몇 개가 채택됐는지 확인합니다. 출력 길이가 달라졌는지도 함께 봅니다.

In [ ]:
counts = dflash[["제안 토큰", "채택 토큰", "채택률 %"]].copy()
counts["단독 출력 토큰 합"] = baseline["출력 토큰 합"]
counts["DFlash 출력 토큰 합"] = dflash["출력 토큰 합"]
counts.round(1)

In [ ]:
comparison[["단독 토큰/초", "DFlash 토큰/초", "단독 전체 p95 ms", "DFlash 전체 p95 ms"]].round(1)

### 9~10분 · 결과 설명

동시 요청 4건인 행에서 초당 출력 토큰 수와 전체 응답 시간 p95가 각각 얼마나 달라졌는지 설명합니다. 처리량이 늘어도 p95가 길어졌다면 두 변화를 모두 말합니다. 시연을 마치면 서버 터미널에서 **Ctrl+C**를 누릅니다.